# ETL da camada silver para camada gold

Esta célula importa todas as bibliotecas necessárias para o processo de ETL.
Aqui são carregados os pacotes para manipulação de dados (pandas), conexão com o banco de dados PostgreSQL (psycopg), controle de mensagens de erro (sys) e tratamento de avisos (warnings).
Ela deve ser executada antes de qualquer outra célula, pois fornece as dependências básicas que serão usadas nas etapas de Extract, Transform e Load.

In [1]:
import pandas as pd
import numpy as np
import psycopg
from psycopg import connect, sql
import sys
import warnings
import math

warnings.filterwarnings('ignore')

# 1. Extract

Esta célula define as configurações de conexão com o banco de dados PostgreSQL e monta a consulta SQL que será usada para extrair os dados.
Ela cria variáveis com credenciais, monta o nome completo da tabela (schema.tabela) e gera a query SELECT * FROM silver.listings, além de preparar a connection string usada na etapa de conexão.

In [2]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA = "silver"
TABLE_NAME = "listings"

TABLE_FULL_NAME = sql.SQL("{}.{}").format(
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(TABLE_NAME)
)

query_object = sql.SQL("SELECT * FROM {}").format(TABLE_FULL_NAME)

connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

Esta célula executa a extração dos dados do banco PostgreSQL.
Ela estabelece a conexão usando as configurações definidas anteriormente, converte o objeto SQL em uma query legível, executa a consulta e carrega o resultado no DataFrame df.
Em caso de falha na conexão ou na leitura, exibe uma mensagem de erro detalhada e encerra o processo.

In [3]:
try:
    print("Estabelecendo conexão...")
    with connect(connection_string) as conn:
        print("Conexão estabelecida.")
        query_string = query_object.as_string(conn)
        print(f"Executando query: {query_string}")
        df = pd.read_sql_query(query_string, conn)
    print("\nDados carregados do banco para o DataFrame com sucesso!")
except psycopg.Error as e:
    print(f"\n--- Ocorreu um erro ao conectar ou ler o banco de dados ---")
    print(f"Erro: {e}")
    sys.exit(1)
except Exception as e:
    print(f"\n--- Ocorreu um erro inesperado ---")
    print(f"Erro: {e}")
    sys.exit(1)


Estabelecendo conexão...
Conexão estabelecida.
Executando query: SELECT * FROM "silver"."listings"

Dados carregados do banco para o DataFrame com sucesso!


Estas células exibem um resumo simples do resultado da extração, mostrando o número total de registros carregados no DataFrame df, as primeiras três tuplas e os tipos de cada dado.
Elas servem para confirmar visualmente que a consulta foi executada com sucesso e quantas linhas foram retornadas do banco.

In [4]:
print(f"Total de linhas carregadas: {len(df)}")


Total de linhas carregadas: 99417


In [5]:
df.head(3)

,id,host_id,name,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,1001254,80014485718,Clean & quiet apt home by the park,False,Madaline,Brooklyn,Kensington,40.64749,-73.97237,False,...,193.0,10,9,2021-10-19,0.21,4.0,6,286,Clean up and treat the home the way you'd like...,True
1,1002102,52335172823,Skylit Midtown Castle,True,Jenna,Manhattan,Midtown,40.75362,-73.98377,False,...,28.0,30,45,2022-05-21,0.38,4.0,2,228,Pet friendly but please confirm with me if the...,True
2,1002403,78829239556,THE VILLAGE OF HARLEM....NEW YORK !,True,Elise,Manhattan,Harlem,40.80902,-73.94190,True,...,124.0,3,0,None,NaN,5.0,1,352,"I encourage you to use my kitchen, cooking and...",True


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99417 entries, 0 to 99416
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              99417 non-null  int64  
 1   host_id                         99417 non-null  int64  
 2   name                            99417 non-null  object 
 3   host_identity_verified          99417 non-null  bool   
 4   host_name                       99417 non-null  object 
 5   neighbourhood_group             99417 non-null  object 
 6   neighbourhood                   99417 non-null  object 
 7   lat                             99417 non-null  float64
 8   long                            99417 non-null  float64
 9   instant_bookable                99417 non-null  bool   
 10  cancellation_policy             99364 non-null  object 
 11  room_type                       99417 non-null  object 
 12  construction_year               

# 2. Transform

Esta célula realiza a padronização dos nomes das colunas do DataFrame para o modelo usado no Data Warehouse.
Ela aplica o dicionário mapa_colunas para renomear os campos extraídos do banco e remove a coluna regras_txt, que não será utilizada nas etapas seguintes de transformação e carga.

In [7]:
mapa_colunas = {
    'id': 'id_anun',
    'host_id': 'id_anfi',
    'host_name': 'nm_anfi',
    'host_identity_verified': 'anfi_verif',
    'calculated_host_listings_count': 'anfi_tot_anun',
    'neighbourhood_group': 'grp_bairro',
    'neighbourhood': 'bairro',
    'lat': 'lat',
    'long': 'long',
    'name': 'nm_anun',
    'instant_bookable': 'res_inst',
    'cancellation_policy': 'pol_cancel',
    'room_type': 'tipo_qto',
    'construction_year': 'ano_constr',
    'minimum_nights': 'min_noites',
    'has_house_rules': 'tem_regras',
    'house_rules': 'regras_txt',
    'last_review': 'dt_ult_rev',
    'price': 'preco',
    'service_fee': 'tx_serv',
    'number_of_reviews': 'tot_rev',
    'reviews_per_month': 'rev_mes',
    'review_rate_number': 'nota_rev',
    'availability_365': 'disp_365'
}

df = df.rename(columns=mapa_colunas)
df = df.drop(columns=['regras_txt'], errors='ignore')


Esta célula define as listas de colunas que compõem cada dimensão e converte a coluna de data da última avaliação (dt_ult_rev) para o tipo datetime.
Essa conversão garante que o campo temporal esteja no formato correto para gerar os atributos de tempo nas etapas seguintes do Transform.

In [8]:
col_aval = 'dt_ult_rev'
cols_anfi = ['nm_anfi', 'anfi_verif', 'anfi_tot_anun']
cols_loc = ['lat', 'long', 'bairro', 'grp_bairro']
cols_prop = ['nm_anun', 'tipo_qto', 'min_noites', 'pol_cancel', 'res_inst', 'ano_constr', 'tem_regras']

df[col_aval] = pd.to_datetime(df[col_aval], errors='coerce')

Esta célula cria o DataFrame df_aval, que representa a dimensão de tempo das últimas avaliações.
Ela seleciona apenas a coluna de data, remove valores nulos e duplicados, e deriva as colunas ano, mes e trimestre a partir de dt_ult_rev, preparando os dados para carga na tabela DIM_ULTIMA_AVALIACAO.

In [9]:
df_aval = df[[col_aval]].copy()
df_aval = df_aval.dropna(subset=[col_aval])
df_aval = df_aval.drop_duplicates(subset=[col_aval])
df_aval['ano'] = df_aval[col_aval].dt.year.astype('Int64')
df_aval['mes'] = df_aval[col_aval].dt.month.astype('Int64')
df_aval['trimestre'] = df_aval[col_aval].dt.quarter.astype('Int64')

Esta célula cria os DataFrames das dimensões de anfitrião, localização e propriedade.
Ela seleciona as colunas correspondentes a cada dimensão, garantindo que cada conjunto contenha apenas valores únicos antes da carga no Data Warehouse.

In [10]:
df_anfi = df[cols_anfi].drop_duplicates().copy()
df_loc = df[cols_loc].drop_duplicates().copy()
df_prop = df[cols_prop].drop_duplicates().copy()


Esta célula realiza a padronização e limpeza dos campos numéricos do DataFrame.
Ela converte todas as colunas de métricas para tipo numérico, substitui valores infinitos por NaN e depois transforma todos os NaN e valores ausentes em None, garantindo que o banco de dados receba NULL corretamente durante a carga.

In [11]:
num_cols = ['rev_mes', 'tot_rev', 'preco', 'tx_serv', 'disp_365', 'nota_rev']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.replace([np.inf, -np.inf], np.nan)
df = df.where(pd.notna(df), None)


Esta célula exibe um resumo da etapa de transformação, mostrando quantos registros foram preparados em cada dimensão e na tabela fato base.
Em seguida, utiliza df.info() para apresentar a estrutura geral do DataFrame principal, permitindo verificar tipos de dados e possíveis valores nulos antes da carga no banco.

In [12]:
print("Registros preparados para carga:")
print(f"DIM_ANFITRIAO: {len(df_anfi)}")
print(f"DIM_LOCALIZACAO: {len(df_loc)}")
print(f"DIM_PROPRIEDADE: {len(df_prop)}")
print(f"DIM_ULTIMA_AVALIACAO: {len(df_aval)}")
print(f"FATO_ANUNCIO (base df): {len(df)}\n\n")

df.info()

Registros preparados para carga:
DIM_ANFITRIAO: 25735
DIM_LOCALIZACAO: 65408
DIM_PROPRIEDADE: 93536
DIM_ULTIMA_AVALIACAO: 2436
FATO_ANUNCIO (base df): 99417


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99417 entries, 0 to 99416
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_anun        99417 non-null  int64         
 1   id_anfi        99417 non-null  int64         
 2   nm_anun        99417 non-null  object        
 3   anfi_verif     99417 non-null  bool          
 4   nm_anfi        99417 non-null  object        
 5   grp_bairro     99417 non-null  object        
 6   bairro         99417 non-null  object        
 7   lat            99417 non-null  float64       
 8   long           99417 non-null  float64       
 9   res_inst       99417 non-null  bool          
 10  pol_cancel     99364 non-null  object        
 11  tipo_qto       99417 non-null  object        
 12  ano_constr   

# 3. Load

Esta célula configura os parâmetros de conexão com o banco de dados do Data Warehouse (dw) e valida se todas as variáveis geradas na etapa de transformação estão disponíveis na memória.
Ela garante que o ambiente esteja pronto antes de iniciar a fase de carga, evitando erros por falta de dados ou variáveis necessárias.

In [13]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA_GOLD = "dw"

connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

for v in ['df', 'df_anfi', 'df_loc', 'df_prop', 'df_aval', 'cols_anfi', 'cols_loc', 'cols_prop', 'col_aval']:
    if v not in globals():
        raise RuntimeError(f"Variável ausente: {v}")


Esta célula executa o script DDL responsável por criar ou recriar as tabelas do schema dw no banco de dados.
Ela lê o arquivo SQL que contém a definição das tabelas e executa o comando dentro de uma conexão com o PostgreSQL, preparando a estrutura necessária para receber os dados na etapa de carga.

In [14]:
try:
    ddl_gold = open('../../data_layer/gold/gold_ddl.sql').read()
except FileNotFoundError:
    print("Erro: Arquivo 'gold_ddl.sql' não encontrado em '../../data_layer/gold/gold_ddl.sql'.")
    sys.exit(1)

with connect(connection_string) as conn:
    with conn.cursor() as cur:
        cur.execute(ddl_gold)


Esta célula realiza a carga da dimensão Anfitrião no schema dw.
Ela insere todos os registros do DataFrame df_anfi na tabela DIM_ANFITRIAO e adiciona uma linha extra com valores nulos para representar o registro “desconhecido”, armazenando sua chave substituta (unknown_anfi_key) para uso posterior na carga da tabela fato.

In [15]:
with connect(connection_string) as conn:
    with conn.cursor() as cur:
        insert_query = sql.SQL("INSERT INTO dw.DIM_ANFITRIAO (nm_anfi, anfi_verif, anfi_tot_anun) VALUES (%s, %s, %s)")
        cur.executemany(insert_query, [tuple(x) for x in df_anfi.to_numpy()])
        cur.execute("INSERT INTO dw.DIM_ANFITRIAO (nm_anfi, anfi_verif, anfi_tot_anun) VALUES (NULL, NULL, NULL) RETURNING srk_anfi")
        unknown_anfi_key = cur.fetchone()[0]


Esta célula insere os dados da dimensão Localização no schema dw.
Ela carrega todos os registros do DataFrame df_loc na tabela DIM_LOCALIZACAO e adiciona um registro adicional com valores nulos para representar a localização “desconhecida”, salvando sua chave substituta (unknown_loc_key) para uso posterior na carga da tabela fato.

In [16]:
with connect(connection_string) as conn:
    with conn.cursor() as cur:
        insert_query = sql.SQL("INSERT INTO dw.DIM_LOCALIZACAO (lat, long, bairro, grp_bairro) VALUES (%s, %s, %s, %s)")
        cur.executemany(insert_query, [tuple(x) for x in df_loc.to_numpy()])
        cur.execute("INSERT INTO dw.DIM_LOCALIZACAO (lat, long, bairro, grp_bairro) VALUES (NULL, NULL, NULL, NULL) RETURNING srk_local")
        unknown_loc_key = cur.fetchone()[0]


Esta célula carrega os dados da dimensão Propriedade no schema dw.
Ela insere os registros do DataFrame df_prop na tabela DIM_PROPRIEDADE e adiciona um registro com valores nulos para representar propriedades desconhecidas, salvando sua chave substituta (unknown_prop_key) que será usada na inserção da tabela fato.

In [17]:
with connect(connection_string) as conn:
    with conn.cursor() as cur:
        insert_query = sql.SQL("INSERT INTO dw.DIM_PROPRIEDADE (nm_anun, tipo_qto, min_noites, pol_cancel, res_inst, ano_constr, tem_regras) VALUES (%s, %s, %s, %s, %s, %s, %s)")
        cur.executemany(insert_query, [tuple(x) for x in df_prop.to_numpy()])
        cur.execute("INSERT INTO dw.DIM_PROPRIEDADE (nm_anun, tipo_qto, min_noites, pol_cancel, res_inst, ano_constr, tem_regras) VALUES (NULL, NULL, NULL, NULL, NULL, NULL, NULL) RETURNING srk_prop")
        unknown_prop_key = cur.fetchone()[0]


Esta célula realiza a carga da dimensão Última Avaliação no schema dw.
Ela insere os registros do DataFrame df_aval na tabela DIM_ULTIMA_AVALIACAO e adiciona um registro com valores nulos para representar avaliações ausentes, armazenando a chave substituta (unknown_aval_key) que será utilizada posteriormente na carga da tabela fato.

In [18]:
with connect(connection_string) as conn:
    with conn.cursor() as cur:
        insert_query = sql.SQL("INSERT INTO dw.DIM_ULTIMA_AVALIACAO (dt_ult_rev, ano, mes, trimestre) VALUES (%s, %s, %s, %s)")
        cur.executemany(insert_query, [tuple(x) for x in df_aval.to_numpy()])
        cur.execute("INSERT INTO dw.DIM_ULTIMA_AVALIACAO (dt_ult_rev, ano, mes, trimestre) VALUES (NULL, NULL, NULL, NULL) RETURNING srk_aval")
        unknown_aval_key = cur.fetchone()[0]


Esta célula faz o mapeamento das chaves substitutas (SRKs) das dimensões para o DataFrame principal.
Ela lê as tabelas dimensionais do banco, realiza os joins com o DataFrame original (df) e substitui valores ausentes pelas chaves “desconhecidas”.
O resultado é o DataFrame df_fato, já com todas as referências dimensionais resolvidas e pronto para ser inserido na tabela fato FATO_ANUNCIO.

In [19]:
with connect(connection_string) as conn:
    df_anfi_com_chaves = pd.read_sql("SELECT * FROM dw.DIM_ANFITRIAO", conn)
    df_loc_com_chaves = pd.read_sql("SELECT * FROM dw.DIM_LOCALIZACAO", conn)
    df_prop_com_chaves = pd.read_sql("SELECT * FROM dw.DIM_PROPRIEDADE", conn)
    df_aval_com_chaves = pd.read_sql("SELECT * FROM dw.DIM_ULTIMA_AVALIACAO", conn)

df_aval_com_chaves['dt_ult_rev'] = pd.to_datetime(df_aval_com_chaves['dt_ult_rev'])

df_m = df.copy()
df_m = pd.merge(df_m, df_anfi_com_chaves.drop_duplicates(subset=cols_anfi), on=cols_anfi, how='left')
df_m = pd.merge(df_m, df_loc_com_chaves.drop_duplicates(subset=cols_loc), on=cols_loc, how='left')
df_m = pd.merge(df_m, df_prop_com_chaves.drop_duplicates(subset=cols_prop), on=cols_prop, how='left')
df_m = pd.merge(df_m, df_aval_com_chaves.drop_duplicates(subset=[col_aval]), on=col_aval, how='left')

df_m['srk_anfi'] = df_m['srk_anfi'].fillna(unknown_anfi_key).astype(int)
df_m['srk_local'] = df_m['srk_local'].fillna(unknown_loc_key).astype(int)
df_m['srk_prop'] = df_m['srk_prop'].fillna(unknown_prop_key).astype(int)
df_m['srk_aval'] = df_m['srk_aval'].fillna(unknown_aval_key).astype(int)

cols_fato = ['disp_365', 'preco', 'tx_serv', 'tot_rev', 'rev_mes', 'nota_rev', 'srk_anfi', 'srk_local', 'srk_aval', 'srk_prop']
df_fato = df_m[cols_fato].copy()


Esta célula realiza a carga final da tabela fato FATO_ANUNCIO no schema dw.
Ela insere todos os registros do DataFrame df_fato, já com as chaves substitutas das dimensões, consolidando os dados no modelo estrela do Data Warehouse.

In [20]:
with connect(connection_string) as conn:
    with conn.cursor() as cur:
        insert_query = sql.SQL("""
            INSERT INTO dw.FATO_ANUNCIO (
                disp_365, preco, tx_serv, tot_rev,
                rev_mes, nota_rev,
                srk_anfi, srk_local, srk_aval, srk_prop
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """)

        cols = ['disp_365','preco','tx_serv','tot_rev','rev_mes','nota_rev',
                'srk_anfi','srk_local','srk_aval','srk_prop']

        def _to_db(v):
            if v is None:
                return None
            if isinstance(v, float) and math.isnan(v):
                return None
            if pd.isna(v):
                return None
            return v

        rows = [
            tuple(_to_db(v) for v in row)
            for row in df_fato[cols].itertuples(index=False, name=None)
        ]

        cur.executemany(insert_query, rows)


Esta célula consulta diretamente o banco de dados para verificar a quantidade de registros inseridos em cada tabela.
Ela exibe o total de linhas carregadas nas dimensões e na tabela fato, funcionando como uma checagem final para confirmar que a etapa de carga foi concluída com sucesso.

In [21]:
with connect(connection_string) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM dw.DIM_ANFITRIAO"); dim_anfi_count = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM dw.DIM_LOCALIZACAO"); dim_loc_count = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM dw.DIM_PROPRIEDADE"); dim_prop_count = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM dw.DIM_ULTIMA_AVALIACAO"); dim_aval_count = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM dw.FATO_ANUNCIO"); fato_count = cur.fetchone()[0]
print(f"Registros carregados no banco:")
print(f"DIM_ANFITRIAO: {dim_anfi_count}")
print(f"DIM_LOCALIZACAO: {dim_loc_count}")
print(f"DIM_PROPRIEDADE: {dim_prop_count}")
print(f"DIM_ULTIMA_AVALIACAO: {dim_aval_count}")
print(f"FATO_ANUNCIO: {fato_count}")


Registros carregados no banco:
DIM_ANFITRIAO: 25736
DIM_LOCALIZACAO: 65409
DIM_PROPRIEDADE: 93537
DIM_ULTIMA_AVALIACAO: 2437
FATO_ANUNCIO: 99417


In [22]:
import pandas as pd
import psycopg
from psycopg import connect
import sys

DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
connection_string = f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}"

queries = {
    1: """SELECT AVG(rev_mes) AS media_avaliacoes_por_mes
          FROM dw.FATO_ANUNCIO AS fato_anuncio;""",

    2: """SELECT SUM(tot_rev) AS soma_total_avaliacoes
          FROM dw.FATO_ANUNCIO AS fato_anuncio;""",

    3: """SELECT COUNT(DISTINCT bairro) AS quantidade_bairros
          FROM dw.DIM_LOCALIZACAO AS dimensao_localizacao;""",

    4: """SELECT COUNT(DISTINCT SRK_anfi) AS quantidade_anfitrioes
          FROM dw.DIM_ANFITRIAO AS dimensao_anfitriao;""",

    5: """SELECT dimensao_ultima_avaliacao.ano AS ano,
                  COUNT(fato_anuncio.SRK_fato_anuncio) AS total_ultimas_avaliacoes
           FROM dw.FATO_ANUNCIO AS fato_anuncio
           JOIN dw.DIM_ULTIMA_AVALIACAO AS dimensao_ultima_avaliacao
             ON fato_anuncio.SRK_aval = dimensao_ultima_avaliacao.SRK_aval
           WHERE dimensao_ultima_avaliacao.ano IS NOT NULL
             AND dimensao_ultima_avaliacao.ano <= 2022
           GROUP BY dimensao_ultima_avaliacao.ano
           ORDER BY dimensao_ultima_avaliacao.ano ASC;""",

    6: """SELECT dimensao_localizacao.grp_bairro AS grupo_bairro,
                  COUNT(fato_anuncio.SRK_fato_anuncio) AS total_anuncios
           FROM dw.FATO_ANUNCIO AS fato_anuncio
           JOIN dw.DIM_LOCALIZACAO AS dimensao_localizacao
             ON fato_anuncio.SRK_local = dimensao_localizacao.SRK_local
           GROUP BY dimensao_localizacao.grp_bairro
           ORDER BY total_anuncios DESC;""",

    7: """SELECT dimensao_ultima_avaliacao.mes AS mes,
                  TO_CHAR(TO_DATE(dimensao_ultima_avaliacao.mes::text, 'MM'), 'Month') AS nome_mes,
                  COUNT(fato_anuncio.SRK_fato_anuncio) AS total_ultimas_avaliacoes
           FROM dw.FATO_ANUNCIO AS fato_anuncio
           JOIN dw.DIM_ULTIMA_AVALIACAO AS dimensao_ultima_avaliacao
             ON fato_anuncio.SRK_aval = dimensao_ultima_avaliacao.SRK_aval
           WHERE dimensao_ultima_avaliacao.mes IS NOT NULL
           GROUP BY dimensao_ultima_avaliacao.mes
           ORDER BY dimensao_ultima_avaliacao.mes ASC;""",

    8: """SELECT dimensao_localizacao.bairro AS bairro,
                  AVG(fato_anuncio.preco) AS media_preco
           FROM dw.FATO_ANUNCIO AS fato_anuncio
           JOIN dw.DIM_LOCALIZACAO AS dimensao_localizacao
             ON fato_anuncio.SRK_local = dimensao_localizacao.SRK_local
           GROUP BY dimensao_localizacao.bairro
           ORDER BY media_preco DESC;""",

    9: """SELECT dimensao_anfitriao.nm_anfi AS nome_anfitriao,
                  SUM(fato_anuncio.tot_rev) AS total_avaliacoes_anfitriao
           FROM dw.FATO_ANUNCIO AS fato_anuncio
           JOIN dw.DIM_ANFITRIAO AS dimensao_anfitriao
             ON fato_anuncio.SRK_anfi = dimensao_anfitriao.SRK_anfi
           GROUP BY dimensao_anfitriao.nm_anfi
           ORDER BY total_avaliacoes_anfitriao DESC;""",

    10: """SELECT dimensao_localizacao.grp_bairro AS grupo_bairro,
                   dimensao_propriedade.tipo_qto AS tipo_quarto,
                   AVG(fato_anuncio.preco) AS media_preco
            FROM dw.FATO_ANUNCIO AS fato_anuncio
            JOIN dw.DIM_LOCALIZACAO AS dimensao_localizacao
              ON fato_anuncio.SRK_local = dimensao_localizacao.SRK_local
            JOIN dw.DIM_PROPRIEDADE AS dimensao_propriedade
              ON fato_anuncio.SRK_prop = dimensao_propriedade.SRK_prop
            GROUP BY dimensao_localizacao.grp_bairro, dimensao_propriedade.tipo_qto
            ORDER BY dimensao_localizacao.grp_bairro, dimensao_propriedade.tipo_qto;""",

    11: """SELECT dimensao_localizacao.grp_bairro AS grupo_bairro,
                   dimensao_propriedade.tipo_qto AS tipo_quarto,
                   AVG(fato_anuncio.rev_mes) AS media_avaliacoes_por_mes
            FROM dw.FATO_ANUNCIO AS fato_anuncio
            JOIN dw.DIM_LOCALIZACAO AS dimensao_localizacao
              ON fato_anuncio.SRK_local = dimensao_localizacao.SRK_local
            JOIN dw.DIM_PROPRIEDADE AS dimensao_propriedade
              ON fato_anuncio.SRK_prop = dimensao_propriedade.SRK_prop
            GROUP BY dimensao_localizacao.grp_bairro, dimensao_propriedade.tipo_qto
            ORDER BY dimensao_localizacao.grp_bairro, dimensao_propriedade.tipo_qto;"""
}

def mostrar_resultados(titulo, df_sql, df_pandas, head_n=10):
    print(f"\n==============================")
    print(f"Consulta {titulo}")
    print("==============================")
    print("\nResultado SQL:")
    print(df_sql.head(head_n) if isinstance(df_sql, pd.DataFrame) else df_sql)
    print("\nResultado Pandas:")
    print(df_pandas.head(head_n) if isinstance(df_pandas, pd.DataFrame) else df_pandas)
    print("==============================\n")

try:
    with connect(connection_string) as conexao:
        # Mapas e dimensoes
        mapa_localizacao = pd.read_sql_query(
            "SELECT SRK_local AS srk_local, bairro, grp_bairro FROM dw.DIM_LOCALIZACAO", conexao
        )
        mapa_propriedade = pd.read_sql_query(
            "SELECT SRK_prop AS srk_prop, tipo_qto FROM dw.DIM_PROPRIEDADE", conexao
        )
        mapa_ultima_avaliacao = pd.read_sql_query(
            "SELECT SRK_aval AS srk_aval, ano, mes FROM dw.DIM_ULTIMA_AVALIACAO", conexao
        )
        dimensao_anfitriao = pd.read_sql_query(
            "SELECT SRK_anfi AS srk_anfi, nm_anfi FROM dw.DIM_ANFITRIAO", conexao
        )

        # Fato principal
        fato_anuncio = pd.read_sql_query(
            """SELECT SRK_fato_anuncio AS srk_fato_anuncio,
                      rev_mes,
                      tot_rev,
                      preco,
                      SRK_local AS srk_local,
                      SRK_aval AS srk_aval,
                      SRK_anfi AS srk_anfi,
                      SRK_prop AS srk_prop
               FROM dw.FATO_ANUNCIO""",
            conexao
        )

        # 1
        sql_1 = pd.read_sql_query(queries[1], conexao)
        pandas_1 = pd.DataFrame({
            "media_avaliacoes_por_mes": [fato_anuncio["rev_mes"].mean()]
        })
        mostrar_resultados(1, sql_1, pandas_1)

        # 2
        sql_2 = pd.read_sql_query(queries[2], conexao)
        pandas_2 = pd.DataFrame({
            "soma_total_avaliacoes": [fato_anuncio["tot_rev"].sum()]
        })
        mostrar_resultados(2, sql_2, pandas_2)

        # 3
        sql_3 = pd.read_sql_query(queries[3], conexao)
        pandas_3 = pd.DataFrame({
            "quantidade_bairros": [mapa_localizacao["bairro"].nunique()]
        })
        mostrar_resultados(3, sql_3, pandas_3)

        # 4
        sql_4 = pd.read_sql_query(queries[4], conexao)
        pandas_4 = pd.DataFrame({
            "quantidade_anfitrioes": [dimensao_anfitriao["srk_anfi"].nunique()]
        })
        mostrar_resultados(4, sql_4, pandas_4)

        # 5
        sql_5 = pd.read_sql_query(queries[5], conexao)
        pandas_5 = (
            fato_anuncio.merge(mapa_ultima_avaliacao[["srk_aval", "ano"]], on="srk_aval", how="left")
                        .query("ano.notna() and ano <= 2022")
                        .groupby("ano", as_index=False)
                        .size()
                        .rename(columns={"size": "total_ultimas_avaliacoes"})
                        .sort_values("ano")
        )
        mostrar_resultados(5, sql_5, pandas_5)

        # 6
        sql_6 = pd.read_sql_query(queries[6], conexao)
        pandas_6 = (
            fato_anuncio.merge(mapa_localizacao[["srk_local", "grp_bairro"]], on="srk_local", how="left")
                        .groupby("grp_bairro", as_index=False)
                        .size()
                        .rename(columns={"size": "total_anuncios"})
                        .sort_values("total_anuncios", ascending=False)
        )
        mostrar_resultados(6, sql_6, pandas_6)

        # 7
        sql_7 = pd.read_sql_query(queries[7], conexao)
        pandas_7 = (
            fato_anuncio.merge(mapa_ultima_avaliacao[["srk_aval", "mes"]], on="srk_aval", how="left")
                        .query("mes.notna()")
                        .groupby("mes", as_index=False)
                        .size()
                        .rename(columns={"size": "total_ultimas_avaliacoes"})
                        .sort_values("mes")
        )
        pandas_7["nome_mes"] = pandas_7["mes"].map({i: pd.Timestamp(2000, i, 1).strftime("%B") for i in range(1, 13)})
        mostrar_resultados(7, sql_7, pandas_7)

        # 8
        sql_8 = pd.read_sql_query(queries[8], conexao)
        pandas_8 = (
            fato_anuncio.merge(mapa_localizacao[["srk_local", "bairro"]], on="srk_local", how="left")
                        .groupby("bairro", as_index=False)["preco"].mean()
                        .rename(columns={"preco": "media_preco"})
                        .sort_values("media_preco", ascending=False)
        )
        mostrar_resultados(8, sql_8, pandas_8)

        # 9
        sql_9 = pd.read_sql_query(queries[9], conexao)
        pandas_9 = (
            fato_anuncio.merge(dimensao_anfitriao, on="srk_anfi", how="left")
                        .groupby("nm_anfi", as_index=False)["tot_rev"].sum()
                        .rename(columns={"tot_rev": "total_avaliacoes_anfitriao", "nm_anfi": "nome_anfitriao"})
                        .sort_values("total_avaliacoes_anfitriao", ascending=False)
        )
        mostrar_resultados(9, sql_9, pandas_9)

        # 10
        sql_10 = pd.read_sql_query(queries[10], conexao)
        pandas_10 = (
            fato_anuncio.merge(mapa_localizacao[["srk_local", "grp_bairro"]], on="srk_local", how="left")
                        .merge(mapa_propriedade[["srk_prop", "tipo_qto"]], on="srk_prop", how="left")
                        .groupby(["grp_bairro", "tipo_qto"], as_index=False)["preco"].mean()
                        .rename(columns={"preco": "media_preco"})
                        .sort_values(["grp_bairro", "tipo_qto"])
        )
        mostrar_resultados(10, sql_10, pandas_10)

        # 11
        sql_11 = pd.read_sql_query(queries[11], conexao)
        pandas_11 = (
            fato_anuncio.merge(mapa_localizacao[["srk_local", "grp_bairro"]], on="srk_local", how="left")
                        .merge(mapa_propriedade[["srk_prop", "tipo_qto"]], on="srk_prop", how="left")
                        .groupby(["grp_bairro", "tipo_qto"], as_index=False)["rev_mes"].mean()
                        .rename(columns={"rev_mes": "media_avaliacoes_por_mes"})
                        .sort_values(["grp_bairro", "tipo_qto"])
        )
        mostrar_resultados(11, sql_11, pandas_11)

except psycopg.Error as e:
    print(f"\nErro de banco: {e}")
    sys.exit(1)
except Exception as e:
    print(f"\nErro inesperado: {e}")
    sys.exit(1)



Consulta 1

Resultado SQL:
   media_avaliacoes_por_mes
0                  1.377313

Resultado Pandas:
   media_avaliacoes_por_mes
0                  1.377313


Consulta 2

Resultado SQL:
   soma_total_avaliacoes
0                2713384

Resultado Pandas:
   soma_total_avaliacoes
0                2713384


Consulta 3

Resultado SQL:
   quantidade_bairros
0                 224

Resultado Pandas:
   quantidade_bairros
0                 224


Consulta 4

Resultado SQL:
   quantidade_anfitrioes
0                  25736

Resultado Pandas:
   quantidade_anfitrioes
0                  25736


Consulta 5

Resultado SQL:
    ano  total_ultimas_avaliacoes
0  2012                        26
1  2013                        75
2  2014                       234
3  2015                      1775
4  2016                      4156
5  2017                      6449
6  2018                     11171
7  2019                     41738
8  2020                      2047
9  2021                      6375

Resul